**_querying from source_**

In [0]:
%sql
select * from pyspark_data.source.products

### **_reading sql table into dataframe_**

In [0]:
df  = spark.sql("select * from pyspark_data.source.products")
display(df)

### # **_upserts_**

In [0]:
from delta.tables import DeltaTable

## using try catch to avoid errors in case if table is not there else creating it
try:
    dlt_obj = DeltaTable.forPath(spark, "/Volumes/pyspark_data/source/dbvolume/products_sink/")

    dlt_obj.alias("tgt").merge(
        df.alias("src"),
        "src.id = tgt.id"
    ).whenMatchedUpdateAll(
        condition="src.updatedDate >= tgt.updatedDate"
    ).whenNotMatchedInsertAll().execute()
    print("Upsert in progress")

except Exception as e:
    print(f"Error: {e}")
    df.write.format("delta").mode("overwrite").save("/Volumes/pyspark_data/source/dbvolume/products_sink/")

In [0]:
%sql
select * from delta.`/Volumes/pyspark_data/source/dbvolume/products_sink`